# Resume + reprocess

Two ways to recover existing analysis directories without redoing everything
from scratch.

  - **Resume** (the pipeline-side feature). Each layer dir keeps a
    `midas_state.h5` ledger. Stages whose outputs already exist are
    skipped. `--from <stage>` re-runs from a specific stage onward.
  - **Reprocess** (gap #6). Re-run `merge_overlaps` + the consolidated
    HDF5 emitter on a finished result dir, e.g. when you want to add
    `MergeMap.csv` to a pre-existing run, or refresh the consolidated
    HDF5 after a bug-fix.


In [ ]:
from pathlib import Path
import json

from midas_ff_pipeline import Pipeline, PipelineConfig
from midas_ff_pipeline.config import LayerSelection
from midas_ff_pipeline.reprocess import reprocess_dir


## 1. Inputs — edit these

In [ ]:
PARAMS = Path('/path/to/Parameters.txt')
ZARR = Path('/path/to/data.MIDAS.zip')
RESULT_DIR = Path('./resume_demo')


## 2. Resume — first run, then a second invocation

The first call processes everything. The second call should detect each
stage's outputs and skip them — visible in the log as `· <stage> — already complete`.

In [ ]:
config = PipelineConfig(
    result_dir=str(RESULT_DIR),
    params_file=str(PARAMS),
    zarr_path=str(ZARR),
    n_cpus=8,
    device='cpu',
    dtype='float64',
    layer_selection=LayerSelection(start=1, end=1),
    resume='auto',
)
Pipeline(config=config).run()
print('---')
Pipeline(config=config).run()   # idempotent — every stage skipped


## 3. Restart from a specific stage

Use `resume='from'` + `resume_from_stage=<name>` to invalidate that stage
and everything after. Equivalent to `midas-ff-pipeline resume <result_dir> --from indexing`.

In [ ]:
config_redo = PipelineConfig(
    result_dir=str(RESULT_DIR),
    params_file=str(PARAMS),
    zarr_path=str(ZARR),
    n_cpus=8,
    device='cpu',
    dtype='float64',
    layer_selection=LayerSelection(start=1, end=1),
    resume='from',
    resume_from_stage='indexing',
)
Pipeline(config=config_redo).run()


## 4. Reprocess — refresh `MergeMap.csv` + consolidated HDF5

Useful for runs that pre-date these features or to refresh after a bug-fix.
Operates on the layer directory directly. Equivalent CLI:
`midas-ff-pipeline reprocess <result_dir>`.

In [ ]:
layer_dir = RESULT_DIR / 'LayerNr_1'
reprocess_dir(layer_dir, n_cpus=8, device='cpu', dtype='float64')

# Quick check: did MergeMap.csv and the consolidated HDF5 land?
for f in ('MergeMap.csv',):
    p = layer_dir / f
    print(f'{f}: {"present" if p.exists() else "MISSING"}  ({p})')
h5s = list(layer_dir.glob('*_consolidated.h5'))
print(f'consolidated HDF5: {h5s}')


## 5. Status snapshot

Same data shown by `midas-ff-pipeline status <result_dir>`.

In [ ]:
pipe = Pipeline(config=config)
print(json.dumps(pipe.status(), indent=2, default=str))
